# Logistic Regression Example (Wine Quality Dataset)

Here it is demonstrated how to use the `LogisticRegression` module from the CMOR-438 library to classify wine quality.
In this example, the Wine Quality dataset is used to train, test, and evaluate the model.

**Goal: Predict whether a wine is High Quality (score ≥ 7) based on its physicochemical properties.**

The binary classification targets are:
- **Class 0:** Low or Mid quality (score 3–6)
- **Class 1:** High quality (score 7–8)

## 1. Setup and Data Loading

Import the necessary modules and load the Wine Quality dataset.
Features are standardised so gradient descent converges efficiently.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
sys.path.insert(0, '../_shared')
from logistic_regression import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

wine = pd.read_csv('../../../data/WineQT.csv').drop(columns=['Id'])
FEATURE_COLS = [c for c in wine.columns if c != 'quality']

print(f"Dataset loaded: {wine.shape[0]} samples, {len(FEATURE_COLS)} features.")
print(f"Quality distribution:\n{wine['quality'].value_counts().sort_index().to_string()}")

## 2. Preprocessing

Create a binary target (High Quality vs rest), standardise features, and split 80/20.

In [ ]:
X = StandardScaler().fit_transform(wine[FEATURE_COLS].values.astype(float))
y_bin = (wine['quality'].values >= 7).astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y_bin, test_size=0.2, random_state=42, stratify=y_bin)

print(f"Training samples: {X_tr.shape[0]}  |  High quality in train: {y_tr.sum()}")
print(f"Test samples:     {X_te.shape[0]}  |  High quality in test:  {y_te.sum()}")

## 3. Train

Train a Logistic Regression model using gradient descent with L2 regularisation.
The model learns to output a probability P(High Quality | features).

In [ ]:
log = LogisticRegression(learning_rate=0.1, n_iterations=600, l2=0.01)
log.fit(X_tr, y_tr)
print(f'Accuracy: {log.accuracy(X_te, y_te):.4f}')

## 4. Results and Visualisation

Two plots are produced:
- **Loss curve** — Binary Cross-Entropy over training iterations; should decrease and plateau
- **Probability distribution** — histogram of predicted P(High Quality) separated by true class; well-separated distributions indicate a confident, accurate model

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(log.loss_history_, color='crimson', lw=1.5)
axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('Binary Cross-Entropy')
axes[0].set_title('Logistic Regression Loss Curve', fontweight='bold')

probs = log.predict_proba(X_te)
for label, color, name in zip([0,1],['steelblue','darkorange'],['Low/Mid','High Quality']):
    axes[1].hist(probs[y_te==label], bins=25, alpha=0.6, color=color, label=name)
axes[1].axvline(0.5, color='red', linestyle='--', lw=1.5, label='Threshold')
axes[1].set_xlabel('P(High Quality)'); axes[1].set_ylabel('Count')
axes[1].set_title('Predicted Probability Distribution', fontweight='bold')
axes[1].legend()
plt.tight_layout(); plt.show()